In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="4"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="4"
import json
from typing import Dict, List, Any
from tqdm import tqdm
from functools import partial

import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch_npu
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
base_model = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
base_dataset = "/data/datasets/Llama-3.2-3B-Instruct-evals"
general_datasets = [
    "Llama-3.2-3B-Instruct-evals__mmlu__details",
    "Llama-3.2-3B-Instruct-evals__gpqa__details",
    "Llama-3.2-3B-Instruct-evals__arc_challenge__details"
]
math_datasets = [
    "Llama-3.2-3B-Instruct-evals__gsm8k__details",
    "/data/datasets/MathInstruct",
]
humaneval_datasets = [
    "evalplus/humanevalplus",
]
magicoder_datasets = [
    "/data/datasets/Magicoder-Evol-Instruct-110K",
]
mbpp_datasets = [
    "evalplus/mbppplus"
]

In [4]:
def preprocess_ultrachat(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str,Any]:
    messages = examples['messages']
    messages = tokenizer.apply_chat_template(messages, tokenize=False)
    return { "inst":messages }

In [5]:
# def preprocess_fn_general(example:Dict[str, Any], tokenizer:AutoTokenizer):
#     multi_turns = tokenizer.encode(example['input_final_prompts'][0])
#     return { "input_ids": [multi_turns] }

def task_preprocess(example:Dict[str, str], tokenizer:AutoTokenizer, task:str="humaneval")->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "mbpp":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"

        python_prefix = 'Write a python function to '
        func_prefix = 'Write a function to '
        if python_prefix in example['prompt']:
            prefix = python_prefix
        elif func_prefix in example['prompt']:
            prefix = func_prefix
        else:
            prefix = ""
        prompt = example['prompt'].replace(prefix, '').strip().capitalize()
        task_prompt = f"""\
{instruction_prefix}
```
{example['code'].split(":")[0].strip()}:
    \"\"\"
    {prompt}
    >>> {example['test_list'][0].replace("assert", "").strip()}
    True
    \"\"\"
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "magicoder":
        return {
            "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
            }
    elif task == "mathinstruct":
        return {
            "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
        }

  

In [6]:
general_datasets = [
    load_dataset(
        base_dataset,
        name=item,
        num_proc=8,
    )['latest'] for item  in general_datasets
]
math_datasets = [
    load_dataset(
        base_dataset,
        name=math_datasets[0],
        num_proc=8,
    )['latest'],
    load_dataset(
        math_datasets[1],
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mathinstruct"),
        num_proc=8,
    )['train']
]
humaneval_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="humaneval"),
        num_proc=8,
    )['test'] for item in humaneval_datasets
]
magicoder_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="magicoder"),
        num_proc=8,
        load_from_cache_file=False,
    )['train'] for item in magicoder_datasets
]
mbpp_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mbpp"),
        num_proc=8,
        load_from_cache_file=False,
    )['test'] for item in mbpp_datasets
]

Map (num_proc=8):   0%|          | 0/111183 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/378 [00:00<?, ? examples/s]

In [7]:
# print(magicoder_datasets[0][4]['inst'])
print(mbpp_datasets[0][5]['inst'])
# print(mbpp_datasets[0][5]['code'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
def square_nums(nums):
    """
    Find squares of individual elements in a list.
    >>> square_nums([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])==[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
    True
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:
```python



In [8]:
print(humaneval_datasets[0][4]['inst'])
# print(humaneval_datasets[0][1]['canonical_solution'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
from typing import List


def mean_absolute_deviation(numbers: List[float]) -> float:
    """ For a given list of input numbers, calculate Mean Absolute Deviation
    around the mean of this dataset.
    Mean Absolute Deviation is the average absolute difference between each
    element and a centerpoint (mean in this case):
    MAD = average | x - x_mean |
    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])
    1.0
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:
```python



In [9]:
# print(general_datasets[2][0]['input_final_prompts'][0])
# print(general_datasets[1][200]['inst'])

In [10]:
# print(math_datasets[0][0]['input_final_prompts'][0])
# print(math_datasets[0][0]['input_correct_responses'])
# print(math_datasets[-1][0]['input_final_prompts'][0])

In [11]:
print(math_datasets[1][1]['inst'])
print(math_datasets[1][1]['output'])

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

How many ways can the letters in the word COMMON be arranged?
Answer Choices: (A) 6 (B) 30 (C) 90 (D) 120 (E) 180<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Let's solve the multi-choice question step by step.
According to the above the # of permutations of 6 letters COMMON out of which 2 O's and 2 M's are identical is 6!2!∗2!=180
The answer is E.


In [12]:
# print(magicoder_datasets[0][3]['inst'])
# print(magicoder_datasets[0][3].keys())
print(magicoder_datasets[0][3]['response'])

This task requires writing of a significant volume of code, which is not fully suitable for a text-based medium. However, I will outline a general solution using Python and scikit-learn. We'll use "CountVectorizer" for bag-of-words model and "TfidVectorizer" for TF-IDF. To handle different languages, we can use 'langdetect' library.

1. Import required libraries
```python
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from langdetect import detect
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
```

2. Load sentence data and labels. For example, if data is stored in a csv format:
```python
data = pd.read_cs

In [13]:
print(len(general_datasets[0]),
      len(general_datasets[1]),
      len(general_datasets[2]),
      len(math_datasets[0]), 
      len(math_datasets[1]),
      len(humaneval_datasets[0]), 
      len(mbpp_datasets[0]), 
      len(magicoder_datasets[0]))

14042 448 1165 1319 262039 164 378 111183


In [14]:
mix_domains = []

samples = 2000

repeat = 1 if len(general_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(general_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'mmlu',
            'task_label': 0,
            'label': 0,
            'outputs': item['input_correct_responses'][0]
        })

repeat = 1 if len(general_datasets[1]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(general_datasets[1]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'gpqa',
            'task_label': 1,
            'label': 0,
            'outputs': item['input_correct_responses'][0]
        })

repeat = 1 if len(general_datasets[2]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(general_datasets[2]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'arc_c',
            'task_label': 2,
            'label': 0,
            'outputs': item['input_correct_responses'][0]
        })

In [15]:
repeat = 1 if len(math_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(math_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'gsm8k',
            'task_label': 3,
            'label': 1,
            'outputs': item['input_correct_responses'][0]
        })

repeat = 1 if len(math_datasets[1]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(math_datasets[1]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mathinstruct',
            'task_label': 4,
            'label': 1,
            'outputs': item['output']
        })


In [16]:
repeat = 1 if len(humaneval_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(humaneval_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'humaneval',
            'task_label': 5,
            'label': 2,
            'outputs': item['canonical_solution']
        })

repeat = 1 if len(mbpp_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(mbpp_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mbpp',
            'task_label': 6,
            'label': 2,
            'outputs': item['code']
        })

repeat = 1 if len(magicoder_datasets[0]) >= samples else 1
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 7,
            'label': 2,
            'outputs': item['response']
        })

In [17]:
with open("/data/lihz/datasets/mix_domains_v1/mix_domains_v1.jsonl", 'w') as f:
    for item in mix_domains:
        f.write(json.dumps(item) + '\n')

In [18]:
with open("/data/lihz/datasets/mix_domains_v1/mix_domains_v1.jsonl", 'r') as f:
    for line in f:
        print(json.loads(line).keys())
        print(json.loads(line)['inputs'])
        print(json.loads(line)['task'])
        print(json.loads(line)['task_label'])
        print(json.loads(line)['label'])
        print(json.loads(line)['outputs'])
        break

dict_keys(['inputs', 'task', 'task_label', 'label', 'outputs'])
<|start_header_id|>user<|end_header_id|>

Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: Which of the following conditions will ensure that angular momentum is conserved? I. Conservation of linear momentum II. Zero net external force III. Zero net external torque
A. I and II only
B. I and III only
C. II and III only
D. III only
Your response should end with "The best answer is [the_answer_letter]" where the [the_answer_letter] is one of A, B, C or D.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The best answer is D.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: A pipe full of air is closed at one end. A standing wave is produced in the pipe, causing the pipe to sound a note. Which of the following is a correct statement about the wave’s proper